# Preprocessing

## 0. Imports & Data loading

In [1]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

import kagglehub, os

from imblearn.over_sampling import SMOTE

sns.set_theme(style="whitegrid")
%matplotlib inline

In [2]:
# Load the dataset using KaggleHub
path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
df = pd.read_csv(os.path.join(path, "creditcard.csv"))

In [3]:
print("Shape:", df.shape)

Shape: (284807, 31)


## 1. Stratified train/test split

In [4]:
# Features and target
X = df.drop(columns=['Class'])
y = df['Class']

In [5]:
# Stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [8]:
print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")
print(f"Fraud ratio - train: {y_train.mean():.4f} | test: {y_test.mean():.4f}")

X_train: (227845, 30) | X_test: (56962, 30)
Fraud ratio - train: 0.0017 | test: 0.0017


`Amount` and `Time` have been normalized with StandardScaler (mean=0, std=1).

V1–V28 are PCA components, already scaled, no transformation needed.

## 2. Scaling features

In [6]:
# Fit scaler on train only, then apply to both
scaler = StandardScaler()
X_train[['Amount', 'Time']] = scaler.fit_transform(X_train[['Amount', 'Time']])
X_test[['Amount', 'Time']] = scaler.transform(X_test[['Amount', 'Time']])

In [7]:
# Rename for clarity
X_train = X_train.rename(columns={'Amount': 'Amount_scaled', 'Time': 'Time_scaled'})
X_test = X_test.rename(columns={'Amount': 'Amount_scaled', 'Time': 'Time_scaled'})

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")
print(f"Fraud ratio — train: {y_train.mean():.4f} | test: {y_test.mean():.4f}")
print(X_train[['Amount_scaled', 'Time_scaled']].describe().round(3))

X_train: (227845, 30) | X_test: (56962, 30)
Fraud ratio — train: 0.0017 | test: 0.0017
       Amount_scaled  Time_scaled
count     227845.000   227845.000
mean           0.000       -0.000
std            1.000        1.000
min           -0.352       -1.998
25%           -0.329       -0.856
50%           -0.264       -0.212
75%           -0.043        0.937
max          102.117        1.641


- 80/20 stratified split -> fraud ratio preserved in both sets.
- No separate validation set: model selection and hyperparameter tuning will use **stratified 5-fold cross-validation** on the training set.
- Final evaluation on the test set is done only once, at the end.

## 3. Class Imbalance (SMOTE)

**Problem**:

The model will be trained on 227,845 transactions, only ~393 of which are fraudulent. It will quickly learn that “classifying everything as legitimate” minimizes its errors and ignore the frauds.

**What SMOTE Does?**

SMOTE = Synthetic Minority Oversampling Technique.

Instead of simply duplicating existing frauds, SMOTE creates new synthetic frauds by interpolating between real frauds.

In practice:
1. It takes a real fraud
2. It finds its K nearest neighbors among the other frauds
3. It generates a new point between these two frauds

The result: the training set goes from 0.17% to 50% fraud and the model learns on balanced classes.

**Why not just duplicate the frauds?**

Simple duplication (naive oversampling) copies exactly the same data points, the model memorizes them without generalizing. SMOTE creates diversity in the minority class.

**Why only on the training set?**

The test set represents reality: 0.17% fraud. If we apply SMOTE to it, we’re measuring performance on an artificial distribution, and our metrics will no longer be meaningful in production.

In [9]:
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

In [10]:
print(f"Before SMOTE - fraud: {y_train.sum()} | legitimate: {(y_train == 0).sum()}")
print(f"After SMOTE  - fraud: {y_train_resampled.sum()} | legitimate: {(y_train_resampled == 0).sum()}")
print(f"New fraud ratio: {y_train_resampled.mean():.4f}")

Before SMOTE - fraud: 394 | legitimate: 227451
After SMOTE  - fraud: 227451 | legitimate: 227451
New fraud ratio: 0.5000


SMOTE worked well. The classes are perfectly balanced.

## 4. Save preprocessed data

In [11]:
# Save preprocessed data into data/processed/
os.makedirs('../data/processed', exist_ok=True)

# Original split (for models using class_weight='balanced')
X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

# Resampled train set (for models using SMOTE)
X_train_resampled.to_csv('../data/processed/X_train_resampled.csv', index=False)
y_train_resampled.to_csv('../data/processed/y_train_resampled.csv', index=False)

print("Datasets saved in data/processed/")

Datasets saved in data/processed/
